# Blip vs EM Shower Labelling — Visual Inspection

This notebook validates the gamma pixel classification introduced in `plot_knn_pid.py`.

Gamma pixels (PDG 22) are split into two sub-classes based on **connected-component analysis** of their (channel, tick) positions within each image:

| Class | Index | Description |
|-------|-------|-------------|
| **mu±** | 0 | Muon / anti-muon |
| **EM** | 1 | Electrons, positrons, and **shower-producing** photons (large γ cluster) |
| **proton** | 2 | Proton |
| **pion** | 3 | Charged pion |
| **blip** | 4 | **Isolated** photon deposit — small γ cluster (Compton, photoelectric) |
| **other** | 5 | Everything else |

Connected components are built over **gamma + e± pixels together**: a gamma pixel whose cluster (counting all co-located EM pixels) exceeds `BLIP_MAX_PIXELS` is labelled **EM**; smaller clusters become **blip**. Including e± ensures that a gamma sitting inside an EM shower is not mis-labelled as a blip just because its immediate gamma neighbours are few.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

# ── Path to feature file (edit as needed) ──────────────────────────────────────
NPZ_PATH = Path("/nfs/data/1/mvicenzi/ml-dune-model/dino_checkpoints/test_logtransform/features_ep10.npz")

# ── Blip classification parameters ────────────────────────────────────────────
BLIP_CONNECT_DIST = 5.0   # max pixel distance (channel/tick) to count as connected
# Clusters with <= this many pixels → blip (isolated photon deposit).
# Clusters larger than this → EM shower (shower-producing photon, grouped with e±).
# 50 px is a physically motivated choice: genuine EM showers start at ~50–100 pixels;
# smaller gamma clusters (including medium-sized Compton electrons) are still blip-like.
BLIP_MAX_PIXELS   = 30

# ── Class definitions (mirroring plot_knn_pid.py) ─────────────────────────────
CLASS_NAMES  = ["mu±", "EM", "proton", "pion", "blip", "other"]
CLASS_COLORS = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#aaaaaa"]
CLS_MU, CLS_EM, CLS_PROTON, CLS_PION, CLS_BLIP, CLS_OTHER = range(6)

LEGEND_HANDLES = [
    mpatches.Patch(color=CLASS_COLORS[c], label=CLASS_NAMES[c])
    for c in range(len(CLASS_NAMES))
]

In [ ]:
print(f"Loading {NPZ_PATH.name} ...")
data      = np.load(NPZ_PATH)
pid       = data["pid_labels"].astype(np.int32)   # PDG code per pixel (0 = no truth)
positions = data["positions"].astype(np.int32)     # [N, 2]  (channel, tick), view-local
charges   = data["charges"][:, 0]                  # [N]  raw ADC
offsets   = data["offsets"]                        # [n_images+1]
n_images  = len(offsets) - 1

print(f"  Images        : {n_images:,}")
print(f"  Total pixels  : {len(pid):,}")
print(f"  Gamma pixels  : {(pid == 22).sum():,}  ({100*(pid==22).mean():.1f}%)")
print(f"  e+/- pixels   : {np.isin(pid,[11,-11]).sum():,}  ({100*np.isin(pid,[11,-11]).mean():.1f}%)")
print(f"  mu± pixels    : {np.isin(pid,[13,-13]).sum():,}")
print(f"  No-truth (0)  : {(pid == 0).sum():,}")

In [ ]:
def classify_gammas_per_image(pdg_img, pos_img, connect_dist, blip_max_pixels):
    """
    Run connected-component analysis on gamma + e± pixels within a single image.
    Cluster size counts all EM pixels (gamma + e±); gamma pixels whose cluster
    exceeds blip_max_pixels are labelled EM shower, smaller ones are blip.

    Returns a dict with arrays indexed over gamma pixels only:
        gamma_local_idx : local pixel indices (into the image slice) that are gamma
        comp_label      : connected component ID per gamma pixel
        comp_size       : size of that component (gamma + e± pixels combined)
        is_blip         : True when the component is a blip (size <= blip_max_pixels)
    """
    gamma_local = np.where(pdg_img == 22)[0]
    elec_local  = np.where(np.isin(pdg_img, [11, -11]))[0]

    empty = dict(gamma_local_idx=gamma_local,
                 comp_label=np.array([], dtype=int),
                 comp_size=np.array([], dtype=int),
                 is_blip=np.array([], dtype=bool))

    if len(gamma_local) == 0:
        return empty

    # Build connected components over gamma + e± together.
    # Gammas are placed first so comp_labels_em[:n_gamma] gives gamma component IDs.
    em_local = np.concatenate([gamma_local, elec_local])
    n_em     = len(em_local)
    em_pos   = pos_img[em_local].astype(float)

    if n_em == 1:
        return dict(gamma_local_idx=gamma_local,
                    comp_label=np.array([0]),
                    comp_size=np.array([1]),
                    is_blip=np.array([True]))

    tree  = cKDTree(em_pos)
    pairs = tree.query_pairs(connect_dist)

    if pairs:
        ra, ca = zip(*pairs)
        r = list(ra) + list(ca)
        c = list(ca) + list(ra)
        adj = csr_matrix((np.ones(len(r), dtype=np.float32), (r, c)), shape=(n_em, n_em))
    else:
        adj = csr_matrix((n_em, n_em), dtype=np.float32)

    _, comp_labels_em = connected_components(adj, directed=False)
    unique, counts    = np.unique(comp_labels_em, return_counts=True)
    size_map          = dict(zip(unique.tolist(), counts.tolist()))

    # Results for gamma pixels only (first len(gamma_local) entries in em_local)
    n_gamma    = len(gamma_local)
    gamma_comp = comp_labels_em[:n_gamma]
    comp_sizes = np.array([size_map[lbl] for lbl in gamma_comp])

    return dict(
        gamma_local_idx = gamma_local,
        comp_label      = gamma_comp,
        comp_size       = comp_sizes,
        is_blip         = comp_sizes <= blip_max_pixels,
    )


def build_global_cls(pid, positions, offsets,
                     connect_dist=BLIP_CONNECT_DIST, blip_max_pixels=BLIP_MAX_PIXELS):
    """
    Assign every pixel a class index 0-5 (or -1 for no-truth pixels).
    Gamma pixels are classified per-image via connected components.
    """
    cls = np.full(len(pid), CLS_OTHER, dtype=np.int32)
    cls[np.isin(pid, [13, -13])]   = CLS_MU
    cls[np.isin(pid, [11, -11])]   = CLS_EM
    cls[pid == 2212]               = CLS_PROTON
    cls[np.isin(pid, [211, -211])] = CLS_PION
    cls[pid == 0]                  = -1   # no truth

    n_images = len(offsets) - 1
    for i in range(n_images):
        start, end = int(offsets[i]), int(offsets[i + 1])
        info = classify_gammas_per_image(
            pid[start:end], positions[start:end], connect_dist, blip_max_pixels
        )
        for li, is_blip in zip(info["gamma_local_idx"], info["is_blip"]):
            cls[start + li] = CLS_BLIP if is_blip else CLS_EM

    return cls


In [ ]:
print(f"Classifying gamma pixels  (connect_dist={BLIP_CONNECT_DIST}, blip_max_pixels={BLIP_MAX_PIXELS}) ...")

# Build global class array and cache per-image gamma info for event selection
global_cls      = np.full(len(pid), CLS_OTHER, dtype=np.int32)
global_cls[np.isin(pid, [13, -13])]   = CLS_MU
global_cls[np.isin(pid, [11, -11])]   = CLS_EM
global_cls[pid == 2212]               = CLS_PROTON
global_cls[np.isin(pid, [211, -211])] = CLS_PION
global_cls[pid == 0]                  = -1

image_info = []   # per-image gamma classification results
for i in range(n_images):
    start, end = int(offsets[i]), int(offsets[i + 1])
    info = classify_gammas_per_image(
        pid[start:end], positions[start:end], BLIP_CONNECT_DIST, BLIP_MAX_PIXELS
    )
    image_info.append(info)
    for li, is_blip in zip(info["gamma_local_idx"], info["is_blip"]):
        global_cls[start + li] = CLS_BLIP if is_blip else CLS_EM

# Summary
n_gamma      = (pid == 22).sum()
n_blip_pix   = ((pid == 22) & (global_cls == CLS_BLIP)).sum()
n_shower_pix = ((pid == 22) & (global_cls == CLS_EM)).sum()

print(f"\n  Total gamma pixels  : {n_gamma:,}")
print(f"  → blip              : {n_blip_pix:,}  ({100*n_blip_pix/n_gamma:.1f}%)")
print(f"  → EM shower gamma   : {n_shower_pix:,}  ({100*n_shower_pix/n_gamma:.1f}%)")
print(f"\nFinal class pixel counts:")
for c, name in enumerate(CLASS_NAMES):
    n = (global_cls == c).sum()
    print(f"  {name:<8} : {n:>10,}  ({100*n/len(pid):.1f}%)")

## 1 — Gamma cluster size distribution

The histogram shows how many pixels each connected gamma cluster contains.  
The red dashed line marks `BLIP_MAX_PIXELS` — clusters to the left are labelled **blip**, those to the right are **EM shower**.

In [ ]:
# Collect one size value per unique cluster (not one per gamma pixel)
all_cluster_sizes = []
for info in image_info:
    if len(info["comp_label"]) == 0:
        continue
    unique_comps, first_idx = np.unique(info["comp_label"], return_index=True)
    all_cluster_sizes.extend(info["comp_size"][first_idx].tolist())

all_cluster_sizes = np.array(all_cluster_sizes)
n_blip_clusters   = (all_cluster_sizes <= BLIP_MAX_PIXELS).sum()
n_shower_clusters = (all_cluster_sizes >  BLIP_MAX_PIXELS).sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: histogram (log-log) ────────────────────────────────────────────────
bins = np.logspace(0, np.log10(all_cluster_sizes.max() + 1), 60)
axes[0].hist(all_cluster_sizes[all_cluster_sizes <= BLIP_MAX_PIXELS],
             bins=bins, color="#9467bd", edgecolor="white", linewidth=0.4,
             label=f"blip ({n_blip_clusters:,} clusters)")
axes[0].hist(all_cluster_sizes[all_cluster_sizes > BLIP_MAX_PIXELS],
             bins=bins, color="#ff7f0e", edgecolor="white", linewidth=0.4,
             label=f"EM shower ({n_shower_clusters:,} clusters)")
axes[0].axvline(BLIP_MAX_PIXELS + 0.5, color="red", linestyle="--", linewidth=2,
                label=f"threshold = {BLIP_MAX_PIXELS} px")
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Cluster size (pixels)", fontsize=12)
axes[0].set_ylabel("Number of clusters", fontsize=12)
axes[0].set_title("Gamma cluster size histogram", fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# ── Right: CDF ───────────────────────────────────────────────────────────────
sorted_sizes = np.sort(all_cluster_sizes)
cdf = np.arange(1, len(sorted_sizes) + 1) / len(sorted_sizes)
axes[1].plot(sorted_sizes, cdf, color="steelblue", linewidth=2)
axes[1].axvline(BLIP_MAX_PIXELS + 0.5, color="red", linestyle="--", linewidth=2,
                label=f"threshold = {BLIP_MAX_PIXELS} px  "
                      f"({100*np.mean(all_cluster_sizes <= BLIP_MAX_PIXELS):.0f}% of clusters)")
for thr, ls in zip([5, 20, 50], [":", "-.", "--"]):
    axes[1].axvline(thr + 0.5, color="gray", linestyle=ls, linewidth=1,
                    label=f"size = {thr}  ({100*np.mean(all_cluster_sizes<=thr):.0f}%)")
axes[1].set_xscale("log")
axes[1].set_xlabel("Cluster size (pixels)", fontsize=12)
axes[1].set_ylabel("Cumulative fraction", fontsize=12)
axes[1].set_title("CDF of gamma cluster sizes", fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1.05)

fig.suptitle(f"Gamma pixel clustering  (connect_dist={BLIP_CONNECT_DIST},  "
             f"n_images={n_images},  n_clusters={len(all_cluster_sizes):,})", fontsize=12)
plt.tight_layout()
plt.show()

print("Fraction of clusters by size threshold:")
for thr in [1, 2, 5, 10, 20, 50, 100]:
    pct = 100 * (all_cluster_sizes <= thr).mean()
    print(f"  <= {thr:4d} px : {pct:5.1f}%  ({(all_cluster_sizes <= thr).sum():,} clusters)")

In [ ]:
def plot_event(ax, img_idx, markersize=3, xlim=None, ylim=None, alpha=0.85):
    """
    Scatter-plot all active pixels in an event using the raw npz coordinates.
    Gray background: log(charge). Colored foreground: class labels.
    xlim/ylim: optional zoom window.
    """
    start, end = int(offsets[img_idx]), int(offsets[img_idx + 1])
    pos = positions[start:end]   # (channel, tick)
    ch  = charges[start:end]
    cls = global_cls[start:end]

    # Gray charge background
    ch_log  = np.log1p(ch)
    ch_norm = (ch_log - ch_log.min()) / (ch_log.max() - ch_log.min() + 1e-8)
    gray    = plt.cm.gray_r(ch_norm * 0.55 + 0.05)
    ax.scatter(pos[:, 0], pos[:, 1], c=gray, s=markersize * 0.4,
               linewidths=0, rasterized=True, alpha=0.5, zorder=1)

    # Class-colored pixels
    for c in range(len(CLASS_NAMES)):
        mask = cls == c
        if not mask.any():
            continue
        ax.scatter(pos[mask, 0], pos[mask, 1],
                   c=CLASS_COLORS[c], s=markersize, linewidths=0,
                   rasterized=True, alpha=alpha, zorder=2)

    ax.invert_yaxis()
    ax.set_xlabel("Channel", fontsize=8)
    ax.set_ylabel("Tick", fontsize=8)
    ax.tick_params(labelsize=7)
    if xlim is not None:
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)


# ── Sort events by content ────────────────────────────────────────────────────
events_shower_only = []
events_blip_only   = []
events_mixed       = []

for i, info in enumerate(image_info):
    if len(info["comp_label"]) == 0:
        continue
    unique_comps, first_idx = np.unique(info["comp_label"], return_index=True)
    sizes = info["comp_size"][first_idx]

    has_shower = (sizes > BLIP_MAX_PIXELS).any()
    has_blip   = (sizes <= BLIP_MAX_PIXELS).any()
    max_shower = int(sizes[sizes > BLIP_MAX_PIXELS].max()) if has_shower else 0
    n_blips    = int((sizes <= BLIP_MAX_PIXELS).sum())

    entry = dict(idx=i, max_shower=max_shower, n_blips=n_blips,
                 n_pix=int(offsets[i+1]) - int(offsets[i]))

    if has_shower and not has_blip:
        events_shower_only.append(entry)
    elif has_blip and not has_shower:
        events_blip_only.append(entry)
    elif has_shower and has_blip:
        events_mixed.append(entry)

events_shower_only.sort(key=lambda e: -e["max_shower"])
events_blip_only.sort(key=lambda e:   -e["n_blips"])
events_mixed.sort(key=lambda e:       -e["max_shower"])

print(f"Events with large showers only : {len(events_shower_only)}")
print(f"Events with blips only         : {len(events_blip_only)}")
print(f"Events with both               : {len(events_mixed)}")
if events_shower_only:
    print(f"  Largest shower in shower-only events: {events_shower_only[0]['max_shower']} px")

## 2 — Full event view

Each dot is an active pixel colored by its class label.  
Gray background encodes log(charge).  
Top row: events containing both blips and EM showers.  
Bottom row: shower-only and blip-only events.

In [ ]:
selected = (
    [e["idx"] for e in events_mixed[:4]]          +
    [e["idx"] for e in events_shower_only[:2]]    +
    [e["idx"] for e in events_blip_only[:2]]
)[:8]

n_cols = 2
n_rows = (len(selected) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.5, n_rows * 4), dpi=200)
axes = axes.flatten()

for ax_i, img_idx in enumerate(selected):
    ax = axes[ax_i]
    plot_event(ax, img_idx, markersize=3)

    start, end = int(offsets[img_idx]), int(offsets[img_idx + 1])
    cls_img = global_cls[start:end]
    n_blip  = (cls_img == CLS_BLIP).sum()
    n_em    = (cls_img == CLS_EM).sum()
    info    = image_info[img_idx]
    n_gamma_clusters = len(np.unique(info["comp_label"])) if len(info["comp_label"]) else 0

    ax.set_title(f"Event {img_idx} | blip px={n_blip}  EM px={n_em}  γ-clusters={n_gamma_clusters}",
                 fontsize=8)

for ax in axes[len(selected):]:
    ax.set_visible(False)

fig.legend(handles=LEGEND_HANDLES, loc="lower center", ncol=6, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Full event view — pixels colored by class label", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3 — Blip gallery (zoom)

Each panel zooms in on a single isolated gamma cluster labelled **blip** (purple).  
The surrounding gray pixels are other active reco hits in the same image.  
Good blips should look like small, compact, isolated islands of purple.

In [ ]:
PAD = 25   # channel/tick padding around each zoomed cluster

blip_examples = []
candidate_events = events_blip_only[:30] + events_mixed[:30]

for entry in candidate_events:
    img_idx = entry["idx"]
    start   = int(offsets[img_idx])
    info    = image_info[img_idx]
    if len(info["gamma_local_idx"]) == 0:
        continue

    pos_img     = positions[start : start + int(offsets[img_idx+1]) - start]
    gamma_local = info["gamma_local_idx"]
    comp_label  = info["comp_label"]
    comp_size   = info["comp_size"]
    is_blip     = info["is_blip"]

    # One entry per unique blip component
    for comp_id in np.unique(comp_label[is_blip]):
        mask   = (comp_label == comp_id)
        g_pos  = pos_img[gamma_local[mask]]
        size   = int(comp_size[mask][0])

        ch_min, ch_max = int(g_pos[:, 0].min()), int(g_pos[:, 0].max())
        tk_min, tk_max = int(g_pos[:, 1].min()), int(g_pos[:, 1].max())

        blip_examples.append(dict(
            img_idx = img_idx,
            size    = size,
            xlim    = (ch_min - PAD, ch_max + PAD),
            ylim    = (tk_min - PAD, tk_max + PAD),
        ))

    if len(blip_examples) >= 20:
        break

# ── Plot ────────────────────────────────────────────────────────────────────────
n_show = min(16, len(blip_examples))
n_cols = 4
n_rows = (n_show + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 3.0), dpi=500)
axes = axes.flatten()

for ax_i, ex in enumerate(blip_examples[:n_show]):
    ax = axes[ax_i]
    plot_event(ax, ex["img_idx"], markersize=12,
               xlim=ex["xlim"], ylim=ex["ylim"], alpha=0.9)
    ax.set_title(f"Ev {ex['img_idx']}  size={ex['size']} px", fontsize=8)

for ax in axes[n_show:]:
    ax.set_visible(False)

fig.legend(handles=LEGEND_HANDLES, loc="lower center", ncol=6, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"Blip gallery — isolated γ clusters  (size ≤ {BLIP_MAX_PIXELS} px)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4 — EM shower gallery (zoom)

Each panel zooms in on a large gamma cluster labelled **EM** (orange).  
Good EM showers should look like extended, branching structures — not compact dots.

In [ ]:
SHOWER_PAD = 40   # larger padding for extended shower structures

shower_examples = []
candidate_events = events_shower_only[:30] + events_mixed[:30]

for entry in candidate_events:
    img_idx = entry["idx"]
    start   = int(offsets[img_idx])
    info    = image_info[img_idx]
    if len(info["gamma_local_idx"]) == 0:
        continue

    pos_img     = positions[start : start + int(offsets[img_idx+1]) - start]
    gamma_local = info["gamma_local_idx"]
    comp_label  = info["comp_label"]
    comp_size   = info["comp_size"]
    is_shower   = ~info["is_blip"]

    # Sort shower components by size (largest first)
    shower_comps = [
        (comp_id, int(comp_size[comp_label == comp_id][0]))
        for comp_id in np.unique(comp_label[is_shower])
    ]
    shower_comps.sort(key=lambda x: -x[1])

    for comp_id, size in shower_comps[:2]:   # at most 2 showers per event
        mask  = (comp_label == comp_id)
        g_pos = pos_img[gamma_local[mask]]

        ch_min, ch_max = int(g_pos[:, 0].min()), int(g_pos[:, 0].max())
        tk_min, tk_max = int(g_pos[:, 1].min()), int(g_pos[:, 1].max())

        shower_examples.append(dict(
            img_idx = img_idx,
            size    = size,
            xlim    = (ch_min - SHOWER_PAD, ch_max + SHOWER_PAD),
            ylim    = (tk_min - SHOWER_PAD, tk_max + SHOWER_PAD),
        ))

    if len(shower_examples) >= 16:
        break

# ── Plot ────────────────────────────────────────────────────────────────────────
n_show = min(16, len(shower_examples))
n_cols = 4
n_rows = (n_show + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.0, n_rows * 3.5), dpi=500)
axes = axes.flatten()

for ax_i, ex in enumerate(shower_examples[:n_show]):
    ax = axes[ax_i]
    plot_event(ax, ex["img_idx"], markersize=6,
               xlim=ex["xlim"], ylim=ex["ylim"], alpha=0.9)
    ax.set_title(f"Ev {ex['img_idx']}  size={ex['size']} px", fontsize=8)

for ax in axes[n_show:]:
    ax.set_visible(False)

fig.legend(handles=LEGEND_HANDLES, loc="lower center", ncol=6, fontsize=9,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f"EM shower gallery — large γ clusters  (size > {BLIP_MAX_PIXELS} px)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5 — Side-by-side: blips and showers in the same event

For events that contain **both** blip clusters and a large EM shower, this section shows the full event (left) alongside zoomed views of the blip region (centre) and the shower region (right).  
This is the most direct visual check that the two types are correctly separated.

In [ ]:
N_MIXED_SHOW = 4   # number of mixed events to display

fig, axes = plt.subplots(N_MIXED_SHOW, 3,
                          figsize=(14, N_MIXED_SHOW * 3.8),
                          gridspec_kw={"wspace": 0.35, "hspace": 0.45})

for row, entry in enumerate(events_mixed[:N_MIXED_SHOW]):
    img_idx = entry["idx"]
    start   = int(offsets[img_idx])
    info    = image_info[img_idx]
    pos_img = positions[start : int(offsets[img_idx + 1])]

    gamma_local = info["gamma_local_idx"]
    comp_label  = info["comp_label"]
    comp_size   = info["comp_size"]
    is_blip     = info["is_blip"]
    is_shower   = ~is_blip

    # Pick one representative blip (smallest isolated one)
    blip_ids  = np.unique(comp_label[is_blip])
    shower_ids = sorted(
        np.unique(comp_label[is_shower]),
        key=lambda c: -int(comp_size[comp_label == c][0])
    )

    blip_id   = blip_ids[0]
    shower_id = shower_ids[0] if len(shower_ids) else None

    blip_pos   = pos_img[gamma_local[comp_label == blip_id]]
    shower_pos = pos_img[gamma_local[comp_label == shower_id]] if shower_id is not None else None

    # ── Left: full event ─────────────────────────────────────────────────────
    plot_event(axes[row, 0], img_idx, markersize=2)
    # Mark blip and shower zoom windows
    def draw_rect(ax, pos, pad, color, label):
        ch0, ch1 = pos[:, 0].min() - pad, pos[:, 0].max() + pad
        tk0, tk1 = pos[:, 1].min() - pad, pos[:, 1].max() + pad
        rect = plt.Rectangle((ch0, tk0), ch1 - ch0, tk1 - tk0,
                              linewidth=1.5, edgecolor=color,
                              facecolor="none", linestyle="--", label=label)
        ax.add_patch(rect)

    draw_rect(axes[row, 0], blip_pos,   PAD,         "#9467bd", "blip zoom")
    if shower_pos is not None:
        draw_rect(axes[row, 0], shower_pos, SHOWER_PAD, "#ff7f0e", "shower zoom")
    axes[row, 0].legend(fontsize=7, loc="upper right")
    axes[row, 0].set_title(f"Event {img_idx} — full view", fontsize=9)

    # ── Centre: blip zoom ────────────────────────────────────────────────────
    b_xlim = (int(blip_pos[:, 0].min()) - PAD, int(blip_pos[:, 0].max()) + PAD)
    b_ylim = (int(blip_pos[:, 1].min()) - PAD, int(blip_pos[:, 1].max()) + PAD)
    plot_event(axes[row, 1], img_idx, markersize=14,
               xlim=b_xlim, ylim=b_ylim, alpha=0.95)
    blip_size = int(comp_size[comp_label == blip_id][0])
    axes[row, 1].set_title(f"Blip zoom  (size={blip_size} px)", fontsize=9)

    # ── Right: shower zoom ───────────────────────────────────────────────────
    if shower_pos is not None:
        s_xlim = (int(shower_pos[:, 0].min()) - SHOWER_PAD,
                  int(shower_pos[:, 0].max()) + SHOWER_PAD)
        s_ylim = (int(shower_pos[:, 1].min()) - SHOWER_PAD,
                  int(shower_pos[:, 1].max()) + SHOWER_PAD)
        plot_event(axes[row, 2], img_idx, markersize=5,
                   xlim=s_xlim, ylim=s_ylim, alpha=0.95)
        shower_size = int(comp_size[comp_label == shower_id][0])
        axes[row, 2].set_title(f"EM shower zoom  (size={shower_size} px)", fontsize=9)
    else:
        axes[row, 2].set_visible(False)

fig.legend(handles=LEGEND_HANDLES, loc="lower center", ncol=6, fontsize=10,
           bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Blip vs EM shower — same event, side-by-side", fontsize=13, y=1.005)
plt.show()

## 6 — Threshold sensitivity

Re-run the classification at three different `BLIP_MAX_PIXELS` thresholds and compare the labelling on the same event to judge where the boundary should be.

In [ ]:
THRESHOLDS  = [10, 20, 30, 50, 100]   # compare old default (10) against physically-motivated values
TEST_EVENTS = [e["idx"] for e in events_mixed[:3]]

fig, axes = plt.subplots(len(TEST_EVENTS), len(THRESHOLDS),
                          figsize=(len(THRESHOLDS) * 4, len(TEST_EVENTS) * 3.8),
                          gridspec_kw={"wspace": 0.3, "hspace": 0.45})

for row, img_idx in enumerate(TEST_EVENTS):
    start, end = int(offsets[img_idx]), int(offsets[img_idx + 1])
    pos_img = positions[start:end]
    ch_img  = charges[start:end]
    pdg_img = pid[start:end]

    for col, thr in enumerate(THRESHOLDS):
        ax = axes[row, col]

        # Rebuild class labels for this threshold
        info = classify_gammas_per_image(pdg_img, pos_img, BLIP_CONNECT_DIST, thr)
        cls_thr = np.full(end - start, CLS_OTHER, dtype=np.int32)
        cls_thr[np.isin(pdg_img, [13, -13])]   = CLS_MU
        cls_thr[np.isin(pdg_img, [11, -11])]   = CLS_EM
        cls_thr[pdg_img == 2212]               = CLS_PROTON
        cls_thr[np.isin(pdg_img, [211, -211])] = CLS_PION
        cls_thr[pdg_img == 0]                  = -1
        for li, is_blip in zip(info["gamma_local_idx"], info["is_blip"]):
            cls_thr[li] = CLS_BLIP if is_blip else CLS_EM

        # Gray charge background
        ch_log  = np.log1p(ch_img)
        ch_norm = (ch_log - ch_log.min()) / (ch_log.max() - ch_log.min() + 1e-8)
        gray    = plt.cm.gray_r(ch_norm * 0.55 + 0.05)
        ax.scatter(pos_img[:, 0], pos_img[:, 1], c=gray, s=1,
                   linewidths=0, rasterized=True, alpha=0.5, zorder=1)

        for c in range(len(CLASS_NAMES)):
            mask = cls_thr == c
            if not mask.any():
                continue
            ax.scatter(pos_img[mask, 0], pos_img[mask, 1],
                       c=CLASS_COLORS[c], s=3, linewidths=0,
                       rasterized=True, alpha=0.85, zorder=2)

        ax.invert_yaxis()
        ax.tick_params(labelsize=6)
        n_blip_px   = (cls_thr == CLS_BLIP).sum()
        n_shower_px = ((pdg_img == 22) & (cls_thr == CLS_EM)).sum()
        label = f"thr={thr}" + (" ← current" if thr == BLIP_MAX_PIXELS else "")
        ax.set_title(f"{label}  blip={n_blip_px} / EM-γ={n_shower_px}", fontsize=8)

        if col == 0:
            ax.set_ylabel(f"Event {img_idx}", fontsize=8)

fig.legend(handles=LEGEND_HANDLES, loc="lower center", ncol=6, fontsize=10,
           bbox_to_anchor=(0.5, -0.01))
fig.suptitle("Threshold sensitivity  (blip_max_pixels varies across columns)",
             fontsize=12, y=1.005)
plt.show()